In [2]:
import pandas as pd

In [3]:
#讀進檔案，客戶ID跟分行代碼用str讀以免前面少0，把'-', ' '設為NaN
df=pd.read_csv('transactions.csv', dtype={'客戶ID':str, '分行代碼':str}, na_values=['-', ' '])

In [4]:
df['客戶ID'].head()

0    0010033
1    0010006
2    0010024
3    0010047
4    0010016
Name: 客戶ID, dtype: object

In [5]:
df.dtypes

客戶ID    object
交易日期    object
交易金額    object
交易類型    object
分行代碼    object
dtype: object

**幫資料做一次全身健檢**

In [7]:
df.shape

(184, 5)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   客戶ID    184 non-null    object
 1   交易日期    173 non-null    object
 2   交易金額    172 non-null    object
 3   交易類型    184 non-null    object
 4   分行代碼    184 non-null    object
dtypes: object(5)
memory usage: 7.3+ KB


In [9]:
df.head(20)

,客戶ID,交易日期,交易金額,交易類型,分行代碼
0,0010033,2024/7/8,800,提款,002
1,0010006,2024/5/15,"5,000",提款,003
2,0010024,2024/1/5,８００,存款,003
3,0010047,2024年5月9日,NaN,提款,007
4,0010016,20-03-2024,"45,000",存款,002
5,0010034,12-01-2024,"1,200",轉帳,003
6,0010032,113/03/11,"NT$128,000",轉帳,002
7,0010003,113/01/15,"45,000",消費,002
8,0010045,2024/2/12,45000,提款,007
9,0010033,113/03/02,"1,200",消費,007


**客戶ID**

In [11]:
df['客戶ID'].value_counts(dropna=False)
#一個ID有幾筆交易

客戶ID
0010008    7
0010033    6
0010020    6
0010022    6
0010035    5
0010037    5
0010023    5
0010015    5
0010006    5
0010029    5
0010047    5
0010012    5
0010046    4
0010019    4
0010049    4
0010001    4
0010041    4
0010039    4
0010050    4
0010018    4
0010038    4
0010045    4
0010032    4
0010016    4
0010005    4
0010027    3
0010010    3
0010017    3
0010013    3
0010009    3
0010002    3
0010048    3
0010003    3
0010044    3
0010024    3
0010007    3
0010011    2
0010028    2
0010014    2
0010026    2
0010025    2
0010021    2
0010031    2
0010030    2
0010040    2
0010034    2
0010036    2
0010042    2
0095457    1
0099078    1
0095140    1
0091233    1
0096397    1
0093165    1
0098946    1
0096879    1
0091624    1
0010004    1
Name: count, dtype: int64

In [12]:
df['客戶ID'].str.len().value_counts()

客戶ID
7    184
Name: count, dtype: int64

In [13]:
#是否全是數字字元？
df['客戶ID'].str.fullmatch(r'\d+').all()

True

**交易日期**

In [15]:
df['交易日期'].value_counts(dropna=False)
#dropna=False → 把Null也算進去

交易日期
NaN           11
2024/7/8       2
2024年2月23日     2
2024-05-01     2
2024-06-18     2
              ..
2024-03-17     1
01-01-2024     1
2024/2/23      1
2024/3/12      1
2024年2月17日     1
Name: count, Length: 157, dtype: int64

In [16]:
#骨架法
df['交易日期'].astype(str).str.replace(r'\d+', 'N', regex=True).value_counts(dropna=False)
#df['交易日期'].astype(str) → 整欄強制轉為字串（NaN原本的型別是float）
#.replace是整格替換， .str.replace是進到每一格的字串裡面做替換
#r'\d+'是正規表達式(regex)，r是Python的raw string標記，告訴Python「反斜線就是反斜線，別當成跳脫字元」
#\d = 任何一個數字字元(0–9)，+ = 前面那個東西「連續出現一次或多次」
#'N' 把每一串連續數字換成一個字母N
#regex=True是明確告訴pandas「第一個參數請當 regex 解讀,不是普通文字」

交易日期
N/N/N     72
N-N-N     69
N年N月N日    32
nan       11
Name: count, dtype: int64

In [17]:
#分流處理日期：用遮罩(mask)把每種格式的列挑出來，各自用各自的方法解析，結果全部匯進同一個新欄位 

#step1 建立空欄
df['交易日期_clean']=pd.NaT #NaT是datetime世界的NaN

In [18]:
#step2 中文處理

#造遮罩
mask=df['交易日期'].str.contains('年', na=False) #把遮罩存進變數（遮罩可以有名字！）
                                               #NaN是Unknown，這樣mask會混進「不是True也不是False」的東西，拿去挑列就報錯
print(mask.sum()) #True有幾個？應該要有32個

#讀+轉換 寫進'交易日期_clean'
#df.loc[row, column]
df.loc[mask, '交易日期_clean']=pd.to_datetime(df.loc[mask, '交易日期'], format='%Y年%m月%d日') #把遮罩蓋上去 → 只顯示True的列

mask.head()

32


0    False
1    False
2    False
3     True
4    False
Name: 交易日期, dtype: bool

In [19]:
df.loc[mask, ['交易日期', '交易日期_clean']].head()

,交易日期,交易日期_clean
3,2024年5月9日,2024-05-09
10,2024年7月17日,2024-07-17
17,2024年2月23日,2024-02-23
18,2024年3月20日,2024-03-20
20,2024年2月23日,2024-02-23


In [20]:
df['交易日期_clean'].notna().sum()

32

In [21]:
#step3 斜線處理(民國 & 西元)

mask_slash=df['交易日期'].str.contains('/', na=False)
mask_slash.sum()   #預期72

72

In [22]:
#分出民國/西元

first=df['交易日期'].str.split('/').str[0]   #取每筆第一段

mask_roc=mask_slash & (first.str.len()==3)  #民國
mask_west=mask_slash & (first.str.len()==4) #西元

mask_roc.sum() + mask_west.sum()

72

In [23]:
#把民國變西元

roc_fixed=(
    (first[mask_roc].astype(int)+1911).astype(str)
    + '/'
    + df.loc[mask_roc, '交易日期'].str.split('/', n=1).str[1]  #n=1：只切一次
)

In [24]:
roc_fixed.head()

6     2024/03/11
7     2024/01/15
9     2024/03/02
11    2024/02/21
13    2024/04/04
Name: 交易日期, dtype: object

In [25]:
#寫進'交易日期_clean'

df.loc[mask_roc, '交易日期_clean']=pd.to_datetime(roc_fixed, format='%Y/%m/%d')
df.loc[mask_west, '交易日期_clean']=pd.to_datetime(df.loc[mask_west, '交易日期'], format='%Y/%m/%d')

In [26]:
df['交易日期_clean'].notna().sum()

104

In [27]:
df['交易日期_clean'].min()

Timestamp('2024-01-02 00:00:00')

In [28]:
df['交易日期_clean'].max()

Timestamp('2024-07-19 00:00:00')

In [29]:
#step4 橫線處理(ISO:2024-05-01 & 歐式:28-04-2024)

mask_hyphen=df['交易日期'].str.contains('-', na=False)
mask_hyphen.sum()

69

In [30]:
#ISO
iso=df['交易日期'].str.split('-').str[0]
mask_iso=mask_hyphen & (iso.str.len()==4)

#EU
eu=df['交易日期'].str.split('-').str[2]
mask_eu=mask_hyphen & (eu.str.len()==4)

mask_iso.sum() + mask_eu.sum()

69

In [31]:
#寫進'交易日期_clean'

df.loc[mask_iso, '交易日期_clean']=pd.to_datetime(df.loc[mask_iso, '交易日期'], format='%Y-%m-%d')
df.loc[mask_eu, '交易日期_clean']=pd.to_datetime(df.loc[mask_eu, '交易日期'], format='%d-%m-%Y')

In [32]:
df.loc[mask_eu, '交易日期_clean'].head()

4    2024-03-20
5    2024-01-12
12   2024-03-03
15   2024-01-30
22   2024-07-18
Name: 交易日期_clean, dtype: datetime64[ns]

In [33]:
print(df['交易日期_clean'].notna().sum())
print(df['交易日期_clean'].isna().sum())
print(df['交易日期_clean'].min())
print(df['交易日期_clean'].max())

173
11
2024-01-01 00:00:00
2024-07-19 00:00:00


In [34]:
df['交易日期_clean'].between('2024-01-01', '2024-07-19')

0       True
1       True
2       True
3       True
4       True
       ...  
179     True
180    False
181     True
182     True
183     True
Name: 交易日期_clean, Length: 184, dtype: bool

In [35]:
df['交易日期_clean'].dt.month.sort_values().unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7., nan])

In [36]:
df['交易日期_clean'].dt.day.sort_values().unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., nan])

In [37]:
df['交易日期_clean'].dt.month.value_counts().sort_index()

交易日期_clean
1.0    28
2.0    15
3.0    30
4.0    31
5.0    27
6.0    18
7.0    24
Name: count, dtype: int64

**交易金額**

In [39]:
#骨架法：看得到「結構的髒」(分隔符、前綴、空白的排列)，看不到「字元的髒」(全形、半形、隱形字元)
df['交易金額'].astype(str).str.replace(r'\d+', 'N', regex=True).value_counts(dropna=False)

交易金額
N           117
N,N          25
 N           14
nan          12
NT$N,N       12
N,N,N         2
NT$N          1
NT$N,N,N      1
Name: count, dtype: int64

In [40]:
df['交易金額'].isna().sum()

12

In [41]:
#step1 全形轉半形：全形世界→半形世界(轉的不只數字)      #.str.translate(str.maketrans('０１２３４５６７８９', '0123456789'))：只轉數字
df['交易金額_clean']=df['交易金額'].str.normalize('NFKC')
df['交易金額_clean'].head(10)

0           800
1         5,000
2           800
3           NaN
4        45,000
5         1,200
6    NT$128,000
7        45,000
8        45000 
9         1,200
Name: 交易金額_clean, dtype: object

In [42]:
#step2 去空格
df['交易金額_clean']=df['交易金額_clean'].str.strip()

In [43]:
#step3 去逗號
df['交易金額_clean']=df['交易金額_clean'].str.replace(',', '')
df['交易金額_clean'].head(10)

0          800
1         5000
2          800
3          NaN
4        45000
5         1200
6    NT$128000
7        45000
8        45000
9         1200
Name: 交易金額_clean, dtype: object

In [44]:
#step4 去NT$
df['交易金額_clean']=df['交易金額_clean'].str.replace('NT$', '', regex=False)
df['交易金額_clean'].head(10)

0       800
1      5000
2       800
3       NaN
4     45000
5      1200
6    128000
7     45000
8     45000
9      1200
Name: 交易金額_clean, dtype: object

In [45]:
#step5 astype轉數值
df['交易金額_clean']=df['交易金額_clean'].astype(float)
df['交易金額_clean'].head()

0      800.0
1     5000.0
2      800.0
3        NaN
4    45000.0
Name: 交易金額_clean, dtype: float64

In [46]:
df['交易金額_clean'].isna().sum()

12

In [47]:
df['交易金額_clean'].value_counts()

交易金額_clean
99999.0      27
800.0        25
128000.0     24
45000.0      23
34500.0      23
5000.0       20
1200.0       15
1234567.0    15
Name: count, dtype: int64

In [48]:
df['交易金額_clean'].min()

800.0

In [49]:
#加總對帳

#刪掉所有不是數字的字元：[^...]是regex的「集合取反」→ 轉dtype
df['交易金額'].str.replace(r'[^0-9０-９]', '', regex=True).astype(float).sum()

26256978.0

In [50]:
df['交易金額_clean'].sum()

26256978.0

**交易類型**

In [52]:
df['交易類型'].value_counts()

交易類型
消費     43
提款     37
轉帳     36
存款     29
 提款     9
 消費     8
 存款     5
 轉帳     5
轉帳　     3
消費　     3
提款　     3
存款　     2
消費      1
Name: count, dtype: int64

In [53]:
df['交易類型_clean']=df['交易類型'].str.strip()

In [54]:
df['交易類型_clean'].value_counts()

交易類型_clean
消費    55
提款    49
轉帳    44
存款    36
Name: count, dtype: int64

In [55]:
df['交易類型_clean'].isnull().sum()

0

**客戶代碼**

In [57]:
df['分行代碼'].str.len().value_counts()

分行代碼
3    184
Name: count, dtype: int64

In [58]:
df['分行代碼'].str.fullmatch(r'\d+').all()

True

In [59]:
df['分行代碼'].value_counts(dropna=False)

分行代碼
007    40
012    40
003    36
001    36
002    32
Name: count, dtype: int64

**清除重複列**

In [61]:
df.duplicated().sum()

3

In [62]:
biz_key=['客戶ID', '交易日期_clean', '交易金額_clean', '交易類型_clean', '分行代碼']
df.duplicated(subset=biz_key).sum()  #subset:只比對這幾欄   #結果會更多，原資料可能因空格不同被判成不同資料

5

In [63]:
mask_dup=df.duplicated(subset=biz_key, keep=False)  #keep=False：重複組的每一份都標 True
                                                    #keep='first'：預設。只標第二份以後(留第一份不標，方便直接 drop)
df[mask_dup].sort_values(biz_key)

,客戶ID,交易日期,交易金額,交易類型,分行代碼,交易日期_clean,交易金額_clean,交易類型_clean
128,0010008,28-04-2024,800,消費,012,2024-04-28,800.0,消費
150,0010008,28-04-2024,800,消費,012,2024-04-28,800.0,消費
91,0010027,113/05/24,128000,轉帳,003,2024-05-24,128000.0,轉帳
141,0010027,113/05/24,128000,轉帳,003,2024-05-24,128000.0,轉帳
132,0010029,NaN,４５０００,提款,003,NaT,45000.0,提款
143,0010029,NaN,45000,提款,003,NaT,45000.0,提款
0,0010033,2024/7/8,800,提款,002,2024-07-08,800.0,提款
58,0010033,2024/7/8,800,提款,002,2024-07-08,800.0,提款
135,0010041,2024-06-18,34500,提款,007,2024-06-18,34500.0,提款
164,0010041,2024-06-18,34500,提款,007,2024-06-18,34500.0,提款


In [64]:
df.loc[[128, 150], '交易類型'].apply(repr)

128     '消費'
150    '消費 '
Name: 交易類型, dtype: object

In [65]:
df_final = df.drop_duplicates()   #只刪3筆完全重複的資料，空格不同造成的重複不能保證不是做兩筆相同交易
len(df_final)

181

**資料驗證**

In [67]:
def validate(df):
    #1 列數：181
    assert len(df)==181, f"列數異常：預期181，實際{len(df)}"
    
    #2 日期空值：交易日期_clean的 NaT數 == 11
    assert df['交易日期_clean'].isna().sum()==11, f"日期空值異常：預期11，實際{df['交易日期_clean'].isna().sum()}"
    
    #3 日期範圍：min >= 2024-01-01 且 max <= 2024-12-31
    assert (df['交易日期_clean'].min()>=pd.Timestamp('2024-01-01')) & (df['交易日期_clean'].max()<=pd.Timestamp('2024-12-31')), '日期範圍異常'
    
    #4 金額對帳：兩欄總額差 < 0.01
    assert abs(df['交易金額'].str.replace(r'[^0-9０-９]', '', regex=True).astype(float).sum() - df['交易金額_clean'].sum()) < 0.01, '金額對帳異常'
    
    #5 金額下限：min > 0
    assert df['交易金額_clean'].min()>0, '金額下限異常'
    
    #6 類型白名單：  # <= (⊆) 用在兩個集合之間意為「是否為子集」
    assert set(df['交易類型_clean'].dropna()) <= {'存款', '提款', '轉帳', '消費'}, f"白名單異常：異常類型{set(df['交易類型_clean'].dropna()) - {'存款', '提款', '轉帳', '消費'}}"
    
    #7 識別碼格式：客戶ID全7碼純數字、分行代碼全3碼純數字
    assert (df['客戶ID'].str.fullmatch(r'\d{7}')).all(), '客戶ID格式異常'
    assert (df['分行代碼'].str.fullmatch(r'\d{3}')).all(), '分行代碼格式異常'
    
    #8 完全重複：df.duplicated().sum() == 0
    assert df.duplicated().sum() == 0, f"完全重複異常：預期0，實際{df.duplicated().sum()}"
    
    #9 業務重複監控：已知待查，報數即可
    biz_key=['客戶ID', '交易日期_clean', '交易金額_clean', '交易類型_clean', '分行代碼']
    print(f"業務重複待查：{df.duplicated(subset=biz_key).sum() - df.duplicated().sum()}筆")

    print('全部檢查通過✓')

In [68]:
validate(df_final)

業務重複待查：2筆
全部檢查通過✓
